# CS2-WM checkpoint inference

Load a checkpoint (default: latest in the chosen run) and run an autoregressive rollout on the validation set.

**Important:** image/GIF outputs are saved to disk (`runs/<run>/inspect/`) and shown via `IPython.display.Image` / `Video` pointing to those files. We deliberately avoid `plt.show()` with embedded base64 figures or `display(fig)` because those bloat the notebook file (each saved frame becomes ~hundreds of KB of base64 inline). Re-running cells overwrites the saved files in place, so the notebook stays small.

## Setup

Configure which run to inspect. Defaults to the latest run with a `latest.pt` checkpoint.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

REPO = Path('..').resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

RUNS_DIR = REPO / 'runs'
DATA_DIR = Path('/opt/dlami/nvme/cs2-data')

# Auto-pick the most recently modified run with latest.pt, unless overridden.
RUN_NAME = None  # e.g. 'full-overnight'; None = auto
if RUN_NAME is None:
    candidates = sorted(
        (p for p in RUNS_DIR.glob('*/latest.pt')),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    assert candidates, f'No latest.pt found under {RUNS_DIR}'
    RUN_DIR = candidates[0].parent
else:
    RUN_DIR = RUNS_DIR / RUN_NAME

CKPT_NAME = 'latest.pt'  # or 'best.pt' / 'step_0010000.pt'
CKPT = RUN_DIR / CKPT_NAME
INSPECT_DIR = RUN_DIR / 'inspect'
INSPECT_DIR.mkdir(parents=True, exist_ok=True)

print('RUN_DIR:', RUN_DIR)
print('CKPT:   ', CKPT, '(exists:', CKPT.exists(), ')')
print('INSPECT_DIR:', INSPECT_DIR)

## Load checkpoint + rebuild model

Always prefer EMA weights for inference (`USE_EMA = True`); fall back to the raw model if the checkpoint predates EMA.

In [ ]:
import torch

from src.action_encoder import NUM_ACTIONS
from src.dataset import CSDataset, collate_diamond
from src.diamond import (
    Denoiser,
    DenoiserConfig,
    InnerModelConfig,
    SigmaDistributionConfig,
)

USE_EMA = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

ckpt = torch.load(CKPT, map_location=DEVICE, weights_only=False)
args = ckpt['args']
preset = args.get('preset_resolved') or {
    'cond_channels': 256, 'base_channels': 64, 'depths': [2, 2, 2, 2], 'attn_depths': [0, 0, 1, 1], 'resize': (36, 64),
}
print('checkpoint step :', ckpt['step'])
print('best_val_mse    :', ckpt.get('best_val_mse'))
print('preset          :', args.get('preset'), preset)
print('has_ema         :', 'ema' in ckpt)

inner = InnerModelConfig(
    img_channels=3,
    num_steps_conditioning=args['cond_frames'],
    cond_channels=preset['cond_channels'],
    depths=list(preset['depths']),
    channels=[preset['base_channels'] * m for m in (1, 2, 4, 8)],
    attn_depths=list(preset['attn_depths']),
    num_actions=NUM_ACTIONS,
    is_upsampler=False,
)
cfg = DenoiserConfig(inner_model=inner, sigma_data=0.5, sigma_offset_noise=0.1, noise_previous_obs=True)
model = Denoiser(cfg).to(DEVICE)
model.setup_training(SigmaDistributionConfig(loc=-1.2, scale=1.2, sigma_min=2e-3, sigma_max=20))

if USE_EMA and 'ema' in ckpt:
    model.load_state_dict(ckpt['ema']['shadow'], strict=True)
    print(f'loaded EMA shadow ({sum(p.numel() for p in model.parameters())/1e6:.1f}M params)')
else:
    model.load_state_dict(ckpt['model'], strict=True)
    print(f'loaded raw model ({sum(p.numel() for p in model.parameters())/1e6:.1f}M params)')
model.eval();

## Build val batch

Re-uses the same fixed indices the trainer uses, so this view aligns with the rollout GIFs already saved under `runs/<run>/rollout/`.

In [ ]:
ROLLOUT_STEPS = 32     # how many frames to predict autoregressively
DENOISE_STEPS = 10     # diffusion sampler steps
VAL_ROWS = 4           # rows in the GIF

val_T = max(args['cond_frames'] + 1 + args['num_autoregressive_steps'],
            args['cond_frames'] + ROLLOUT_STEPS)
val_ds = CSDataset(
    data_path=DATA_DIR, split='val', T=val_T, stride=args['stride'],
    resize=tuple(preset['resize']), mode='diamond',
)
print(f'val: {len(val_ds)} windows / {len(val_ds.samples)} clips, val_T={val_T}')

rng = torch.Generator().manual_seed(123)
val_indices = torch.randperm(len(val_ds), generator=rng)[: args['val_batch_size']].tolist()
print('val indices:', val_indices)
batch = collate_diamond([val_ds[i] for i in val_indices]).to(DEVICE)

## Multi-step rollout (saves GIF to disk)

Output is saved to `runs/<run>/inspect/inspect_rollout.gif` (overwritten each run) and displayed via `Video` pointing to disk. The notebook file itself stays tiny.

In [ ]:
import time
from src.visualize import rollout_autoregressive, save_rollout_gif

n = args['cond_frames']
t0 = time.perf_counter()
pred = rollout_autoregressive(model, batch, num_steps=ROLLOUT_STEPS, num_denoising_steps=DENOISE_STEPS)
dt = time.perf_counter() - t0
gt = batch.obs[:, n : n + ROLLOUT_STEPS]
mse_per_step = ((pred - gt) ** 2).mean(dim=(0, 2, 3, 4)).cpu().tolist()
print(f'rollout {ROLLOUT_STEPS} steps in {dt:.1f}s')
print('mse per step:', [f'{m:.4f}' for m in mse_per_step])

gif_path = INSPECT_DIR / 'inspect_rollout.gif'
save_rollout_gif(pred, gt, gif_path, fps=10, max_rows=VAL_ROWS)
print('saved →', gif_path)

In [ ]:
from IPython.display import Image
# IPython renders GIFs from disk without re-encoding into the notebook,
# so the .ipynb file stays small even after re-runs.
Image(filename=str(gif_path))

## Single-step val grid (PSNR + |diff|)

Same metrics as `train.py`'s val-every cycle. Saves a PNG to disk; the cell shows it via `Image`.

In [ ]:
from src.visualize import run_validation
import matplotlib
matplotlib.use('Agg')  # save-only; no inline matplotlib output → notebook stays small

metrics = run_validation(
    denoiser=model,
    val_batch=batch,
    out_dir=INSPECT_DIR,
    step=int(ckpt['step']),
    num_denoising_steps=DENOISE_STEPS,
    max_rows=VAL_ROWS,
    save_png=True,
)
print(f"val_mse={metrics['val_mse']:.4f}  val_psnr={metrics['val_psnr_db']:.2f} dB")
print('png →', metrics['val_image_path'])
Image(filename=metrics['val_image_path'])

## Browse training-time rollouts (light-weight)

All rollout GIFs the trainer already saved live in `runs/<run>/rollout/`. Walking that folder is much faster than re-running the model.

In [ ]:
rollout_dir = RUN_DIR / 'rollout'
gifs = sorted(rollout_dir.glob('rollout_step_*.gif'))
print(f'found {len(gifs)} saved rollouts')
for g in gifs[-3:]:
    print(g.name)
    display(Image(filename=str(g)))